# Day 1 exercise: news website summarizer

A variation on the day 1 summarizer that points at The Guardian's international front page.
The system prompt asks the model to skip navigation and boilerplate, and to split out news
and announcements into their own section when the page has any.

In [ ]:
from openai import OpenAI
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv(override=True)

client = OpenAI()

In [ ]:
url = "https://www.theguardian.com/international"

response = requests.get(url)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")
website_content = soup.get_text(separator="\n")

# Remove unnecessary blank lines
website_content = "\n".join(
    line.strip()
    for line in website_content.splitlines()
    if line.strip()
)

# Limit the amount of text sent to the LLM
website_content = website_content[:12000]

In [ ]:
system_prompt = """
You are a professional website analyst.

Your job is to:
- Ignore navigation menus, headers, footers, cookie banners, and repetitive text.
- Focus on the actual content.
- Produce a concise Markdown summary.
- If the website contains news or announcements, summarize them separately.
- Keep the tone friendly and slightly humorous.
"""

user_prompt = f"""
Please analyze the following website.

Website Content:
----------------
{website_content}

Provide:

# Website Summary

A short overview.

# Key Points

- Bullet points

# News / Announcements

Only include this section if applicable.
"""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt},
]

In [ ]:
completion = client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=messages
)

display(Markdown(completion.choices[0].message.content))